# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Oscar Soria Corral | Kathryn Day   <br>
Url: https://github.com/osoria80/03MIAR-Algoritmos-de-optimizacion/tree/main/Proyecto<br>
Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:
Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible.

(*) La respuesta es obligatoria





                                        

# (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




**Sin restricciones:**

Para calcular el total de posibilidades de ordenación de las tomas, debemos calcular sus posibles permitaciones, sin posibilidad de repetir tomas, que viene dado por la fórmula $P = n!$. Teniendo en cuenta que nuestro número de tomas és 30 (n = 30):

$$P = 30! = 265252859812191058636308480000000$$


In [16]:
import math
p = math.factorial(30)
print(p)

265252859812191058636308480000000


# Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)


**Primera opción: Listas anidadas** *(Desestimada)*


Para representar la solucion, optamos por utilizar una lista de lista, donde cada sublista representa un día de grabación, y cada entero dentro de cada sublista es el índice de una toma. Puede haber tantas listas como días posibles (máximo 30) y dentro de cada sublista entre 1 y 6 tomas.

$$Solución = [[t_{1,1},t_{1,2},..,t_{1,h}],[t_{2,1},..,t_{2,i}],..,[t_{k,1},..,t_{k,j}]]$$

Dónde: $0 < h,i,j \leqslant 6, 5 \leqslant k \leqslant 30$

Beneficios de esta estructura:


*   Sencilla de interpretar: Tantos días como listas, y tantas tomas como elementos dentro de la sublista. Interpretable a simple vista.
*   Fácil acceso a actores por día: Identificar la toma por su índice, nos permite un cálculo muy sencillo y eficiente del coste por día, siendo este una operación 'OR' de todas las máscaras con indice en la sublista.
* Cálculo de coste total: siguienod el punto anterior, el coste total será la suma de los costes de la lista principal.
* Operabilidad: Acceso sencillo por indexación y operabilidad optimizada con listas.


**Segunda opción: Una sola lista de 30 elementos**


Tras diseñar la estrategia, nos damos cuenta que podemos fijar que cada día (menos el último) se hagan con el máximo de sesiones (6). Por lo que para identificar los días, no hace falta separar en sublistas, sino que los indices nos indican a qué día pertenecen.

$$Solución=[t_1,t_2,..,t_k]$$
Dónde: $0 \leqslant k \leqslant 30$


# Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?


(*)¿Es un problema de maximización o minimización?

Siendo $X_k$ el número de actores presentes en cada día y $k$ los días de rodaje. La función objeivo será el mínimo de la suma de los actores por día.

El coste diario $X_k$, se calculará como una codificación binária de 10 bits, dónde cada bit representa a un actor.

$Bit_n=FALSE \rightarrow \text{Actor n ausente}$<BR>
$Bit_n=TRUE \rightarrow \text{Actor n presente}$
$$X_k:\{1,..,10\}\rightarrow\{1,0\}$$

Por lo tanto, nuestra función objetivo será:

$$Objetivo = Mín X_1 + X_2 + .. + X_k$$


Es un problema de minimización. Queremos que el número de actores presentes cada día sea el mínimo posible, de esta forma, el coste diário será también el mínimo, y consigo el coste total.

In [17]:
#Definimos una función para calcular el coste del día
def coste_dia(tomas):
  mask_dia = 0
  for toma in tomas:
    mask_toma = int(''.join(df.loc[str(toma)].astype(int).astype(str)), 2)
    mask_dia |= mask_toma
  #print(f"Coste del día: {bin(mask_dia).count('1')}")
  return bin(mask_dia).count('1')

#Definimos una función para calcular el coste total
def coste_total(solucion):
  coste = 0
  dias = (len(solucion) + 6 - 1) // 6
  #print(f"Días de rodaje: {dias}")
  for dia in range(dias):
    if dia == dias: coste += coste_dia(solucion[dia*6:])
    else: coste += coste_dia(solucion[dia*6:(dia+1)*6])
  #print(f"Coste total: {coste}")
  return coste



# Diseña un algoritmo para resolver el problema por fuerza bruta

Antes de resolverlo, fijamos las restricciones y definimos una estrategia.

**Restricciones:**
* Máximo 6 tomas por día
* Que se hagan todas las tomas

**Estrategia:**

Definimos que la estrategia a seguir: Hacer el máximo de tomas pemritidas por día.

De esta manera reducimos el nº de días, y por tanto el coste total. Por ello, podemos asumir que todos los días tendrán 6 tomas, exceptuando el último día, el cual se grabarán el restante de las tomas.

**Fuerza bruta:**

Generaremos todas las permutaciones posibles de las 30 sesiones para calcular su coste. Nos quedaremos con la permutación con el menor coste.

Para poder comprobar el código, trabajaremos con una versiones reducida del probelma.


In [18]:
#Realizamos la carga de datos
import pandas as pd
df_ini = pd.read_csv('Datos problema doblaje(30 tomas, 10 actores).csv', index_col=0, header=1).iloc[:30,:10]
df_ini = df_ini.astype(int)

#Utilizamos un dataset más pequeño para hacer comprobaciones
df=df_ini.iloc[:6,:10]
display(df)

,1,2,3,4,5,6,7,8,9,10
Toma,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0
2,0,0,1,1,1,0,0,0,0,0
3,0,1,0,0,1,0,1,0,0,0
4,1,1,0,0,0,0,1,1,0,0
5,0,1,0,1,0,0,0,1,0,0
6,1,1,0,1,1,0,0,0,0,0


In [19]:
#Generamos la primera solución -> Sesiones ordenadas
solucion_ini = [i for i in range(1, len(df)+1)]
coste_min = float('inf') #Inicializamos el valor del coste mínimo
solucion_min = [] #Inicializamos el resultado mínimo

#Definimos la función para generar todas las permutaciones
def permutaciones(solucion, inicio=0):
  global coste_min, solucion_min
  #Comprobamos si esa permutación es la de mínimo coste
  if inicio == len(solucion):
    c = coste_total(solucion)
    if c < coste_min:
      coste_min = c
      solucion_min = solucion[:]
    return

  for i in range(inicio, len(solucion)):
    # Ponemos solucion[i] en la posición 'inicio'
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]
    #Llamamos a 'permutaciones' para permutar el resto de valores
    permutaciones(solucion, inicio + 1)
    # Deshacemos el swap (backtrack) para realizar la siguiente permutación
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]

  return solucion

permutaciones(solucion_ini)
print(f"Mejor solución: {solucion_min}\nCoste: {coste_min}")



Mejor solución: [1, 2, 3, 4, 5, 6]
Coste: 7


# Calcula la complejidad del algoritmo por fuerza bruta

Primero analizamos el coste de la función recursiva 'permutaciones':


*   Para cada solución posible, la función se va a llamar $n$ veces. Para generar los n elementos de la solución.
*   Para calcular todas las permutaciones, la función se va a llamar $n!$ veces. Para generar todas las permutaciones posibles.
*   El resto de operaciones tienen un coste constantes.

El coste computacional de 'permutaciones' es $O(n!)$.


Ahora analizamos el coste de la función 'coste_total':


*   Para cada llamada, se recorre la lista de n elementos 1 vez.
*   El resto de operaciones tienen un coste constante.

Teniendo en cuenta que la función 'coste_total' se llama en cada permutación completa, obtenemos el coste computacional total:

$$\text{Coste computacional: }O(n!·n)$$
<br>
Es un coste extremadamente elevado. Solo para 10 tomas, ya tenemos aproximadamente 36 millones de operaciones. Solo con aumentar 5 tomas más (15 tomas), aumentamos a 13 billones de operaciones. Por lo que para dimensiones mayores, como nuestro problema inicial (30 tomas), tendríamos $2.65\times10^{33}$ operaciones, algo inviable de computar.




# (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

En este apartado se presentan dos opciones:
* **Opción 1:** Branch and bound. (Descatrado tras ejecución y análisis)
* **Opción 2:** Algoritmo genético.

## Branch and Bound

**Opción 1: Branch and Bound** (*Desetimada*)


Para solucionar un problema de este tipo con una solución exacta, un algoritmo de ramificación y poda (Branch & Bound) sería la mejor opción.
En esencia, sería el mismo algoritmo de fuerza bruta, pero añadiendo una cota inferior que permite podar ramas enteras sin explorarlas.

Definiremos nuestro criterio de poda como el siguiente:


$$\text{Coste parcial (días ya completados) + Cota inferior (días restantes)} \geq \text{Coste min → Podar: esta rama nunca mejorará la solución actual}$$

Para tener un algoritmo que mejore de manera notable la ocomplejidad compiutaciones, debemos definir una buena cota inferior, que sea rápida de calcular y intentar no sobreestimar el coste real de la rama.

Definimos nuestra cota inferior:


*   En el mejor caso posible, las tomas restantes de ordenar se agruparían perfectamente para minimizar actores (solución objetivo).
*   La cota asume el escenario más optimista posible:
* En cada día futuro, el número de actores es el mínimo de las tomas restantes.
* Eso quiere decir, que en el mejor de los caso, el coste del día+1 será el de la toma con menos actores de las tomas restantes.

De esta manera no sobreestimamos las ramas ya que estamos siendo muy optimistas con las soluciones futuras. Así, la cota generalmente será más pequeña o igual al coste real.

Tras ejecutar y analizar el algoritmo de ramificación y poda, hemos visto que su complejidad computacional sigue sin ser ejecutable y por tanto, no nos puede dar una solución a nuestro probelma.






In [20]:
#Definimos una función para calcular la cota inferior
def cota_inferior (solucion, inicio):
  #Calculamos el coste de los días ya completados
  if inicio % 6 != 0:
    coste_parcial = coste_total(solucion[:inicio - (inicio % 6)])
  else:
    coste_parcial = coste_total(solucion[:inicio])

  #Calculamos el coste de las tomas del día actual
  if inicio % 6 != 0:
    tomas_dia_actual = solucion[inicio - (inicio % 6):inicio]
    coste_dia_actual = coste_dia(tomas_dia_actual)
  else:
    coste_dia_actual = 0

  #Generamos una lista con las tomas restantes
  tomas_restantes = solucion[inicio:]
  #Ordenamos esa lista de menor a mayor coste para elegir la de menor coste
  costes_tomas = []
  for t in tomas_restantes:
    mask_toma = int(''.join(df.loc[str(t)].astype(int).astype(str)), 2)
    costes_tomas.append(bin(mask_toma).count('1'))
  costes_tomas.sort(reverse=True)
  #costes_tomas.sort()
  #Calculamos el coste más optimista de los siguientes días
  coste_futuro = 0
  for i in range(0, len(costes_tomas), 6):
    coste_futuro += costes_tomas[i]

  coste_rama = coste_parcial + coste_dia_actual + coste_futuro
  return coste_rama


In [21]:
#Definimos el algoritmo de branch and bound
def branch_and_bound(solucion, inicio=0):
  global coste_min, solucion_min, nodos_exp, nodos_pod
  nodos_exp += 1
  #Realizamos la poda si la cota inferior de la rama es mayor al coste mínimo obtenido
  if cota_inferior(solucion, inicio) >= coste_min:
    nodos_pod += 1
    return
  #A partir de aquí utilizamos el algoritmo de fuerza bruta para generar las las permutaciones
  if inicio == len(solucion):
    c = coste_total(solucion)
    if c < coste_min:
      coste_min = c
      solucion_min = solucion[:]
    return

  for i in range(inicio, len(solucion)):
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]
    branch_and_bound(solucion, inicio + 1)
    solucion[inicio], solucion[i] = solucion[i], solucion[inicio]

In [22]:
#Utilizamos un dataset más pequeño para hacer comprobaciones
df=df_ini.iloc[:8,:10]
#display(df)

#Inicializamos la solución optima
coste_min = float('inf') #Inicializamos el valor del coste mínimo
solucion_min = [] #Inicializamos el resultado mínimo

#Inicializamos el contaje de nodos explorados y podas
nodos_exp = 0
nodos_pod = 0

#Ejecutamos el algoritmo
solucion_ini = [i for i in range(1, len(df)+1)]
branch_and_bound(solucion_ini)
print(f"Mejor solución: {solucion_min}\nCoste: {coste_min}")
print(f"Nodos explorados: {nodos_exp}")
print(f"Nodos podados: {nodos_pod}")
p = round(nodos_pod/nodos_exp,3)
print(f"Fracción podada: {p}")

Mejor solución: [1, 2, 3, 4, 5, 6, 7, 8]
Coste: 12
Nodos explorados: 13794
Nodos podados: 8525
Fracción podada: 0.618


## Algoritmo genético

**Opción 2: Algoritmo genético** *(Elegido)*

Se propone utilizar un algoritmo genético para obtener una solución no exacta, pero cerca del óptimo, que sea ejecutable en un tiempo coherente y para tamaños, como mínimo, como los propuestos en el problema.

Para su diseño, definimos la estrategia a seguir:
* **Mutación:** Mutaremos intercambiando posiciones. Como todas las tomas parecen solo una vez, no podemos cambiar valores (repetiríamos tomas). Definimos intercambiar solo 2 tomas para no destruir mucho la estructura del padre.

* **Cruce:** Debemos tener en cuenta no repetir valores. Segmentaremos la estructura del padre 1, y rellenaremos lo faltante con los valores del padre 2.

* **Selección:** Para conseguir un equilibrio entre generar diversidad y que no domine una solución temprana, utilizamos una selección por torneo con 5 individuos. Seleccionaremos aleatoriamente 5 candidatos para quedarnos con el de menor coste.

Con respecto ala estructura de los datos:

En primera instancia, cometemos el error de trabajar solo con los indviduos y calcular su coste cada vez que se requiera. Esto afectaba muy notablemente en el coste computacional del algoritmo, teniendo que acceder a los datos
de manera recurrente.

Para ello, trabajamos en conjunto individuo-coste, realizando una nueva lista con tuplas para evitar ese costoso proceso y calcular el coste asociado a un individuo solo 1 vez cuando sea necesario.


Finalmente, debemos definir los parámetros del algoritmo:
* **Población:** 50 individuos. Encontramos un equilibro entre una población pequeña, que converjería muy rápido en mínimos locales, y una población grande, que sería excesivamente costosa computacionalmente. Tras varias pruebas, establecemos este parámetro.

* **Generaciones:** 100 generaciones. Buscamos tener un coste computacional bajo, y tras realizar algunas pruebas, no mejoramos los resultado por aumentar las generaciones. De esta manera reducimos el coste computacional y convergemos más rápido.

* **Tasa de mutación:** 15% de individuos mutados. Fijamos una tasa algo elevada ya que la mutación es suave y no altera excesivamente la estructura.

* **Elitismo:** 2 individuos. Aseguramos que el mejor coste nunca empeora pasando los 2 mejores individuos íntegramente a la segunda generación.

In [23]:
import random

#Inicializmaos los parámetros del algoritmo genético
n_poblacion = 50
n_generaciones = 100

#Definimos una función para inicializar la población de manera aleatoria
def individuo_aleatorio():
    individuo = list(range(1, len(df) + 1))
    random.shuffle(individuo)
    return individuo
def poblacion_inicial():
    return [individuo_aleatorio() for i in range(n_poblacion)]

#Definimos la función de selección de individuos -
def seleccion(poblacion_con_coste):
    candidatos = random.sample(poblacion_con_coste, 5)
    mejor = min(candidatos, key=lambda x: x[1])
    return mejor[0]

#Definimos una función para cruzar individuos y generar hijos
def cruce(padre1, padre2):
  n = len(padre1)
  a = random.randint(1, n - 1)  #Fijamos un punto de corte aleatorio
  hijo = [None]*n
  hijo[:a] = padre1[:a]
  elem_rep = set(padre1[:a])
  elem_no_rep = [x for x in padre2 if x not in elem_rep]
  for i in range(a, n):
      hijo[i] = elem_no_rep[i - a]
  return hijo

#Definimos una función para mutar individuos
def mutacion(individuo):
    individuo = individuo[:]
    i, j = random.sample(range(len(individuo)), 2)
    individuo[i], individuo[j] = individuo[j], individuo[i]
    return individuo


In [24]:
#Definimos el algoritmo genético
def algoritmo_genetico():
  #Inicializamos población, mejor individuo y su coste
  poblacion = poblacion_inicial()
  poblacion_con_coste = [(individuo, coste_total(individuo)) for individuo in poblacion]
  poblacion_con_coste.sort(key=lambda x: x[1])
  mejor_solucion, mejor_coste = poblacion_con_coste[0]
  mejor_solucion = mejor_solucion[:]

  #Ordenamos la población de mejor a peor coste
  for generacion in range(n_generaciones):
    #Actualizamos el mejor global si hay mejora
    if poblacion_con_coste[0][1] < mejor_coste:
      mejor_coste = poblacion_con_coste[0][1]
      mejor_solucion = poblacion_con_coste[0][0][:]

    #Generamos una nueva generación vacía - Los dos mejores pasan directamente
    nueva_poblacion = [poblacion_con_coste[0], poblacion_con_coste[1]]

    #Generamos los nuevos individuos de la nueva generación
    while len(nueva_poblacion) < n_poblacion:
      #Seleccionamos los padres para reproducirse
      padre1 = seleccion(poblacion_con_coste)
      padre2 = seleccion(poblacion_con_coste)
      #Generamos los hijos a partir de los padres seleccionados
      hijo = cruce(padre1, padre2)
      if random.random() < 0.15: hijo = mutacion(hijo) #Mutamos solo un 15% de los hijos
      nueva_poblacion.append((hijo, coste_total(hijo)))

    poblacion_con_coste = sorted(nueva_poblacion, key=lambda x: x[1])
  return mejor_solucion, mejor_coste

In [25]:
%%time
#Utilizamos un dataset más pequeño para hacer comprobaciones
df=df_ini.iloc[:30,:10]
#display(df)

#Ejecutamos el algoritmo
solucion, coste = algoritmo_genetico()
print(f"Mejor solución: {solucion}\nCoste = {coste}")

Mejor solución: [20, 26, 7, 27, 22, 9, 4, 5, 3, 16, 6, 15, 14, 18, 24, 17, 23, 19, 1, 29, 2, 21, 11, 13, 12, 8, 30, 28, 25, 10]
Coste = 30
CPU times: user 18.7 s, sys: 55.6 ms, total: 18.7 s
Wall time: 18.9 s


**Análisis:** Procedemos a comprobar cuan lejos estamos de la solución óptima global con la ayuda de un solver.

In [26]:
#Generamos un solver para comparar los resultados.
from pulp import *

def solver_pulp(df):
    n_tomas = len(df)
    n_actores = len(df.columns)
    tomas_por_dia = 6
    n_dias = n_tomas // tomas_por_dia
    necesita = df.astype(int).values

    prob = LpProblem("Doblaje", LpMinimize)

    # Variable: x[t][d] = 1 si la toma t se graba el día d
    x = [[LpVariable(f'x_{t}_{d}', cat='Binary')
          for d in range(n_dias)] for t in range(n_tomas)]

    # Variable: y[d][a] = 1 si el actor a trabaja el día d
    y = [[LpVariable(f'y_{d}_{a}', cat='Binary')
          for a in range(n_actores)] for d in range(n_dias)]

    # Objetivo: minimizar actores convocados
    prob += lpSum(y[d][a] for d in range(n_dias) for a in range(n_actores))

    # Cada toma se graba exactamente un día
    for t in range(n_tomas):
        prob += lpSum(x[t][d] for d in range(n_dias)) == 1

    # Cada día tiene exactamente 6 tomas
    for d in range(n_dias):
        prob += lpSum(x[t][d] for t in range(n_tomas)) == tomas_por_dia

    # Si la toma t está en el día d y necesita al actor a, el actor debe estar presente
    for t in range(n_tomas):
        for a in range(n_actores):
            if necesita[t][a] == 1:
                for d in range(n_dias):
                    prob += y[d][a] >= x[t][d]

    prob.solve(PULP_CBC_CMD(msg=0, timeLimit=60))

    coste_optimo = int(value(prob.objective))
    print(f"Estado: {LpStatus[prob.status]}")
    print(f"Coste mínimo encontrado: {coste_optimo}")
    return coste_optimo

In [27]:
#Realizamos la comprobación empírica de la diferencia entre nuestro algoritmo genético y la solución del solver.
df=df_ini.iloc[:30,:10]
n_runs = 10

coste_optimo = solver_pulp(df_ini)
costes_ga = [algoritmo_genetico()[1] for _ in range(n_runs)]

gaps = [((c - coste_optimo) / coste_optimo) * 100 for c in costes_ga]

print(f"Coste óptimo:  {coste_optimo}")
print(f"Costes GA:     {costes_ga}")
print(f"Media:         {sum(costes_ga)/n_runs:.2f}")
print(f"Mejor:         {min(costes_ga)}")
print(f"Peor:          {max(costes_ga)}")
print(f"Gap medio:     {sum(gaps)/n_runs:.2f}%")
print(f"Gap mejor:     {min(gaps):.2f}%")

Estado: Optimal
Coste mínimo encontrado: 27
Coste óptimo:  27
Costes GA:     [30, 31, 30, 31, 29, 28, 30, 30, 30, 30]
Media:         29.90
Mejor:         28
Peor:          31
Gap medio:     10.74%
Gap mejor:     3.70%


**Análisis de resultados**
* **Gap medio:** 10% es un buen resultado, la diferencia de la media de costes con el óptimo global nos indica cuán lejos estamos del óptimo.
* **Mejor caso:** En el caso más favorable, solo estamos a un 3.7% de la solución óptima. Esto nos indica que el espacio de búsqeuda se está explorando correctamente.
* **Estabilidad:** 8 de 10 ejecuciones nos dieron un coste 30, lo que nos indica que el algoritmo es bastante estable y no hay grandes diferencias entre ejecuciones.
* **Rango:** Acorde con la estabilidad, el rango de soluciones es bastante estrecho [28-31], solo 3 unidades entre el mejor y el peor caso.

# (*)Calcula la complejidad del algoritmo

## Branch and bound


**Complejidad de generar el arbol de recursión:**

En este caso tenemos la misma complejidad que en el algoritmo por fuerza bruta: $O(n!)$

<br>

**Complejidad de 'cota_inferior' en cada nodo:**

El coste derivado de la ordenación de las tomas restantes, en el peor de los casos es de: $O(n·log(n))$

<br>

**Coste total sin poda:**

Es la multiplicación de ambos costes: $O(n·nlog(n))$

Es un coste mayor que en la fuerza bruta. Ahora debemos tener en cuenta la poda.

<br>

**Coste con poda:**

En este caso, si el algoritmo poda una fracción 'p' de ramas, esto afectará al coste de generar el arbol de recursión. Siendo ahora: $$O((n·(1-p))!·nlog(n))$$

En particular, con este algoritmo, estamos obteniendo fracciones podadas del 70% aproxiamadamente, lo cual es un buen valor que permite mejorar de manera notable la complejidad compiutacional con respecto a la fuerza bruta.

Sin embargo, sigue siendo inviable para tamaños del problema a partir de las 15-20 tomas.


## Algoritmo genético

En este caso, debemos tener en cuenta los siguientes parámetros:
* n = Tamaño del problema
* P = Población
* G = generaciones

**Coste poblacional:**
El coste de cada generación viene dado por las siguientes funciones predominantes:
* Calculo de los costes iniciales: $O(P·n)$
* Ordenar inicialmente lo P individuos: $O(P·logP)$
<br> El coste de generar la población inicial es:
$$O(P·n)$$

**Coste generacional:**
Para generar todas las generaciones, se repite el coste poblacionalen cada iteración. Por lo tanto, el coste general del algoritmo es:
$$O(G·P·n)$$

Nuestro algoritmo tiene un coste polinomial que, aunque es elevado, es totalmente manejable para el tamaño de problema propuesto.

# Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

In [28]:
import numpy as np
new_data = np.random.randint(0, 2, size=(30, 10))
m = new_data.shape[0]
indices = [str(i) for i in range(1, m+1)]
df = pd.DataFrame(new_data, index=indices)
display(df.head())

,0,1,2,3,4,5,6,7,8,9
1,0,0,1,0,1,1,0,1,1,1
2,0,0,1,0,0,1,1,1,1,1
3,0,1,1,0,1,0,1,1,1,1
4,1,0,1,0,0,1,1,1,0,0
5,1,1,1,1,1,1,1,0,1,1


# Aplica el algoritmo al juego de datos generado

In [29]:
#Ejecutamos el algoritmo
random_solucion, random_coste = algoritmo_genetico()
print(f"Mejor solución: {random_solucion}\nCoste: {random_coste}")

Mejor solución: [2, 12, 4, 19, 1, 13, 14, 26, 8, 21, 17, 18, 7, 9, 15, 20, 30, 29, 5, 3, 11, 28, 22, 23, 25, 6, 24, 27, 16, 10]
Coste: 42


# Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

1. **Material sobre Branch and Bound (PDF).**
Campos, J. (s. f.). Branch and Bound [Material didáctico]. Universidad de Zaragoza. Recuperado de https://webdiis.unizar.es/~jcampos/ab/material/6-BranchBound.pdf

2. **Material sobre programación lineal entera (PDF).**
UPC (s. f.). Programació Lineal Entera i Problemes de PLE [Material docente]. Universitat Politècnica de Catalunya. Recuperado de https://www-eio.upc.edu/teaching/mdio1/MIOPD_PLE.pdf

3. **Organvides, D. R. (2023).** Algoritmos genéticos: teoría y ejercicios resueltos [Apuntes docentes]. Universidad de Sevilla. Recuperado de
https://personal.us.es/drorganvidez/wp-content/uploads/2023/01/ALGORITMOS-GENETICOS-teoria-y-ejercicios-resueltos.pdf

4. **Universidad de Granada. (s. f.).** Tema 06 – Algoritmos Genéticos I [Material docente del curso de Bioinformática]. Recuperado de
https://sci2s.ugr.es/sites/default/files/files/Teaching/GraduatesCourses/Bioinformatica/Tema%2006%20-%20AGs%20I.pdf

5. **Algoritmos genéticos – explicación y ejemplo [Video].** (s. f.). YouTube. Recuperado de
https://www.youtube.com/watch?v=Pj6uQpEkgxc

6. **Pandas — API Reference.**
pandas development team. (2026). API reference — pandas documentation (Versión 3.0.1). PyData. Recuperado de https://pandas.pydata.org/docs/reference/index.html

7. **NumPy Reference.**
NumPy Developers. (2025). NumPy reference manual (v2.4). NumPy. Recuperado de https://numpy.org/doc/stable/reference/


Se ha utilizado inteligencia artificial (Claude - Sonnet 4.6 y GPT-4.1) como herramienta de apoyo para:

* Consultas generales sobre NumPy y Pandas.
*   Consultas sobre programación en Python para la generación y comparación de las máscaras de bits.
* Consultas sobre generación de combinatorias en Python.
*   Corrección de errores en el código generado.
*   Generación del solver y comparación de resultados.


El contenido final, así como las decisiones técnicas y conclusiones, son responsabilidad exclusiva de los autores.

# Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño



**Generar la solución inicial con un algoritmo voraz:**

Ahora, el algoritmo genera una solución incial aleatoria, desconociendo el coste de la misma, por lo que el algoritmo parte de 0.

Si quisieramos mejorar el rendimiento del algoritmo genético
, podríamos generar una solución incial mediante un algoritmo voraz, de esta manera, el algoritmo genético parte de una base con mejor coste y sería capaz de converger más rápido en mejores soluciones.

<br>

**Utilizar un criterio de parada**

Enfocado a tamños de problema mayores, podemos implementar una parada con criterio de convergencia para evitar seguir computando aunque hayamos encontrado un valor aceptable.
